# WtDtPorter 架构

```mermaid
graph TD
    subgraph "外部调用层 (Python/C#)"
        A["<b>外部应用</b><br/>调用C接口, 实现回调函数"]
    end

    subgraph "WtDtPorter 动态库"
        direction LR
        
        subgraph "<b>API接口层</b>"
            B(<b>WtDtPorter.h/.cpp</b>):::api_layer
            B_Desc["<b>作用:</b> 暴露C语言API<br/>作为外部调用的唯一入口<br/>将所有请求转发给WtDtRunner"]
        end

        subgraph "<b>核心协调层</b>"
            C(<b>WtDtRunner.h/.cpp</b>):::runner_layer
            C_Desc["<b>作用:</b> 系统总指挥 (单例)<br/>1. 初始化所有核心模块 (DataManager等)<br/>2. 管理扩展模块的生命周期<br/>3. 持有并触发外部回调函数"]
        end

        subgraph "<b>扩展桥接层</b>"
            D(<b>PorterDefs.h</b>):::defs_layer
            D_Desc["<b>作用:</b> 定义回调函数类型<br/>是C++核心与外部回调之间的'契约'"]
            E(<b>ExpParser.h/.cpp</b>):::bridge_layer
            E_Desc["<b>作用:</b> '代理'行情解析器<br/>将内部订阅请求<br/>转换为外部回调"]
            F(<b>ExpDumper.h/.cpp</b>):::bridge_layer
            F_Desc["<b>作用:</b> '代理'数据存储器<br/>将内部数据转储请求<br/>转换为外部回调"]
        end
    end

    %% --- 关系连线 ---
    A -- "调用API" --> B
    B -- "转发请求" --> C
    
    C -- "创建并管理" --> E & F

    E -- "将内部调用<br/>(如subscribe)" --> C
    F -- "将内部调用<br/>(如dumpHisBars)" --> C
    
    C -- "触发外部回调" --> A

    B -- "包含定义" --> D
    E -- "包含定义" --> D
    F -- "包含定义" --> D
    C -- "使用" --> E & F

    %% 样式定义
    classDef api_layer fill:#c9daf8,stroke:#333,stroke-width:2px
    classDef runner_layer fill:#d9ead3,stroke:#333,stroke-width:3px
    classDef bridge_layer fill:#fce5cd,stroke:#333,stroke-width:2px
    classDef defs_layer fill:#fff2cc,stroke:#333,stroke-width:2px
    classDef build_system fill:#eeeeee,stroke:#666,stroke-width:1px,stroke-dasharray: 5 5
```

# 调用外部代码接口协议 PorterDefs.h

定义了 WtDtPorter 与外部脚本语言（如 Python）之间的回调函数名称：供 WtDtPorter（C++）反向调用外部代码。
- **FuncParser\*** 回调：自定义行情源接入WonderTrader
  - `FuncParserEvtCallback`：typedef void\(PORTER_FLAG \*FuncParserEvtCallback\)\(WtUInt32 evtId, const char* id\);
    - 用于接收**生命周期事件**。C++核心会通过这个回调通知外部程序
      - evtId：事件类型，包括
          - `EVENT_PARSER_INIT` (1): Parser初始化事件：当Parser完成初始化时触发
          - `EVENT_PARSER_CONNECT` (2): Parser连接事件：当Parser成功连接到数据源时触发
          - `EVENT_PARSER_DISCONNECT` (3): Parser断开连接事件：当Parser与数据源断开连接时触发
          - `EVENT_PARSER_RELEASE` (4): Parser释放事件：当Parser释放资源时触发
      - id：Parser的唯一标识符，用于区分不同的Parser实例
  - `FuncParserSubCallback`：typedef void\(PORTER_FLAG \*FuncParserSubCallback\)\(const char* id, const char* fullCode, bool isForSub\);
    - 用于接收**行情订阅/退订请求**，当C++内部需要某个合约的行情时，`ParserAdapter`会触发这个回调。
      - id：Parser的唯一标识符，用于区分不同的Parser实例
      - fullCode：完整的合约代码，包含交易所前缀
      - isForSub：订阅标志，true表示订阅操作，false表示退订操作
- **FuncDump\*** 回调：数据自定义外部转储
  - `FuncDumpBars`、`FuncDumpTicks`、`FuncDumpOrdQue`、`FuncDumpOrdDtl`、`FuncDumpTrans`
  - 分别用于接收 **K线**、**Tick**、**委托队列**、**逐笔委托**、**逐笔成交**数据的批量转储任务
  - 当 StateMonitor 判断到盘后处理时间时，DataManager 会触发数据转储流程，WtDtCore 就会将数据通过这些回调函数到外部程序

# 扩展数据转储器 ExpDumper.h/cpp
```cpp
class ExpDumper : public IHisDataDumper
```
实现历史数据转储接口IHisDataDumper，支持K线、Tick、委托队列、委托明细、逐笔成交等多种数据类型的转储
- 参考 [Includes/note.ipynb/数据管理接口层/数据写入 IDataWriter.h/历史数据存储接口类 IHisDataDumper](../Includes/note.ipynb)

**成员**：
- `std::string	_id`：转储器唯一标识符，用于在回调函数中识别转储器实例

**全局声明**：
获取全局 `WtDtRunner` 单例的引用，该单例在 [./WtDtPorter.cpp](./WtDtPorter.cpp) 中定义
```cpp
extern WtDtRunner& getRunner();
```

## 方法

### 转储历史K线数据 dumpHisBars
```cpp
/**
 * @brief 转储历史K线数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param period K线周期，如"m1"、"m5"、"day"等
 * @param bars K线数据数组指针
 * @param count K线数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将K线数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的K线转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisBars(const char* stdCode, const char* period, WTSBarStruct* bars, uint32_t count)
{
	// 调用WtDtRunner的dumpHisBars方法，将转储器ID作为第一个参数传递
	// 便于回调函数识别数据来源和执行相应的存储逻辑
	return getRunner().dumpHisBars(_id.c_str(), stdCode, period, bars, count);
}
```

### 转储历史Tick数据 dumpHisTicks
```cpp
/**
 * @brief 转储历史Tick数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param ticks Tick数据数组指针
 * @param count Tick数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将Tick数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的Tick转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisTicks(const char* stdCode, uint32_t uDate, WTSTickStruct* ticks, uint32_t count)
{
	// 调用WtDtRunner的dumpHisTicks方法，将转储器ID作为第一个参数传递
	// 支持外部回调函数根据转储器ID执行不同的存储策略
	return getRunner().dumpHisTicks(_id.c_str(), stdCode, uDate, ticks, count);
}
```

### 转储历史委托队列数据 dumpHisOrdQue
```cpp
/**
 * @brief 转储历史委托队列数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 委托队列数据数组指针，包含买卖盘口队列信息
 * @param count 委托队列数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数重写了IHisDataDumper接口的dumpHisOrdQue方法，
 * 将委托队列数据转储请求转发给WtDtRunner进行处理。
 * 委托队列数据主要用于Level2行情分析。
 */
virtual bool dumpHisOrdQue(const char* stdCode, uint32_t uDate, WTSOrdQueStruct* items, uint32_t count) override;
```

### 转储历史委托明细数据 dumpHisOrdDtl
```cpp
/**
 * @brief 转储历史委托明细数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 委托明细数据数组指针
 * @param count 委托明细数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将委托明细数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的委托明细转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisOrdDtl(const char* stdCode, uint32_t uDate, WTSOrdDtlStruct* items, uint32_t count)
{
	// 调用WtDtRunner的dumpHisOrdDtl方法，将转储器ID作为第一个参数传递
	// 便于WtDtRunner进行转储器实例的识别和管理
	return getRunner().dumpHisOrdDtl(_id.c_str(), stdCode, uDate, items, count);
}
```

### 转储历史逐笔成交数据 dumpHisTrans
```cpp
/**
 * @brief 转储历史逐笔成交数据
 * @param stdCode 标准合约代码，格式为"交易所.合约代码"
 * @param uDate 交易日期，格式为YYYYMMDD
 * @param items 逐笔成交数据数组指针
 * @param count 逐笔成交数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将逐笔成交数据转储请求转发给WtDtRunner处理。
 * WtDtRunner会调用外部注册的逐笔成交转储回调函数来完成实际的存储操作。
 */
bool ExpDumper::dumpHisTrans(const char* stdCode, uint32_t uDate, WTSTransStruct* items, uint32_t count)
{
	// 调用WtDtRunner的dumpHisTrans方法，将转储器ID作为第一个参数传递
	// 支持多个转储器同时工作时的识别和管理
	return getRunner().dumpHisTrans(_id.c_str(), stdCode, uDate, items, count);
}
```

# 扩展行情解析器 ExpParser.h/cpp
```cpp
class ExpParser : public IParserApi
```
实现行情解析器接口 IParserApi，支持初始化、连接、订阅、退订等基本操作
- 参考 [Includes/note.ipynb/行情解析 IParserApi.h/行情解析器接口 IParserApi](../Includes/note.ipynb)

**成员**：
- `std::string _id`：解析器唯一标识符，用于在回调函数中识别解析器实例
- `IParserSpi* m_sink`：回调接口指针，用于接收行情数据和事件
- `IBaseDataMgr* m_pBaseDataMgr`：基础数据管理器指针，用于访问合约、交易所等基础信息

**全局声明**：
获取全局 `WtDtRunner` 单例的引用，该单例在 [./WtDtPorter.cpp](./WtDtPorter.cpp) 中定义
```cpp
extern WtDtRunner& getRunner();
```

## 方法

### 初始化解析器 init
```cpp
/**
 * @brief 初始化解析器
 * @param config 配置参数，包含解析器的初始化配置信息
 * @return bool 初始化成功返回true
 * 
 * 该函数将初始化请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的初始化回调函数，通知外部模块进行初始化操作。
 * 当前实现总是返回true，表示初始化成功。
 */
bool ExpParser::init(WTSVariant* config)
{
	// 调用WtDtRunner的parser_init方法，将解析器ID传递给回调函数
	// 外部模块根据解析器ID进行相应的初始化操作
	getRunner().parser_init(_id.c_str());
	return true;  // 总是返回true，表示初始化成功
}
```

### 释放解析器资源 release
```cpp
/**
 * @brief 释放解析器资源
 * 
 * 该函数将释放请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的释放回调函数，通知外部模块清理资源。
 * 外部模块应在此回调中关闭连接、释放内存等清理操作。
 */
void ExpParser::release()
{
	// 调用WtDtRunner的parser_release方法，通知外部模块释放资源
	// 外部模块根据解析器ID识别需要释放的解析器实例
	getRunner().parser_release(_id.c_str());
}
```

### 连接到数据源 connect
```cpp
/**
 * @brief 连接到数据源
 * @return bool 连接成功返回true
 * 
 * 该函数将连接请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的连接回调函数，通知外部模块建立与数据源的连接。
 * 当前实现总是返回true，表示连接请求已发送。
 */
bool ExpParser::connect()
{
	// 调用WtDtRunner的parser_connect方法，通知外部模块建立连接
	// 外部模块根据解析器ID识别需要连接的数据源
	getRunner().parser_connect(_id.c_str());
	return true;  // 总是返回true，表示连接请求已发送
}
```

### 断开与数据源的连接 disconnect
```cpp
/**
 * @brief 断开与数据源的连接
 * @return bool 断开成功返回true
 * 
 * 该函数将断开连接请求转发给WtDtRunner处理。
 * WtDtRunner会触发外部注册的断开连接回调函数，通知外部模块关闭与数据源的连接。
 * 当前实现总是返回true，表示断开连接请求已发送。
 */
bool ExpParser::disconnect()
{
	// 调用WtDtRunner的parser_disconnect方法，通知外部模块断开连接
	// 外部模块根据解析器ID识别需要断开连接的数据源
	getRunner().parser_disconnect(_id.c_str());
	return true;  // 总是返回true，表示断开连接请求已发送
}
```

### 查询连接状态 isConnected
```cpp
/**
 * @brief 查询连接状态
 * @return bool 已连接返回true，未连接返回false
 * 
 * 该函数重写了IParserApi接口的isConnected方法。
 * 当前实现始终返回true，表示解析器处于连接状态。
 * 实际的连接状态由外部数据源维护。
 */
virtual bool isConnected() override { return true; }  // 始终返回true
```

### 订阅合约行情 subscribe
```cpp
/**
 * @brief 订阅合约行情
 * @param setCodes 要订阅的合约代码集合
 * 
 * 该函数将订阅请求转发给WtDtRunner处理。
 * 遍历合约集合，对每个合约代码调用WtDtRunner的订阅方法。
 * WtDtRunner会触发外部注册的订阅回调函数，通知外部模块向数据源发送订阅请求。
 */
void ExpParser::subscribe(const CodeSet& setCodes)
{
	// 遍历合约代码集合，对每个合约进行订阅
	for(const auto& code : setCodes)
		// 调用WtDtRunner的parser_subscribe方法，传递解析器ID和合约代码
		// 外部模块根据这些信息向数据源发送订阅请求
		getRunner().parser_subscribe(_id.c_str(), code.c_str());
}
```

### 退订合约行情 unsubscribe
```cpp
/**
 * @brief 退订合约行情
 * @param setCodes 要退订的合约代码集合
 * 
 * 该函数将退订请求转发给WtDtRunner处理。
 * 遍历合约集合，对每个合约代码调用WtDtRunner的退订方法。
 * WtDtRunner会触发外部注册的退订回调函数，通知外部模块向数据源发送退订请求。
 */
void ExpParser::unsubscribe(const CodeSet& setCodes)
{
	// 遍历合约代码集合，对每个合约进行退订
	for (const auto& code : setCodes)
		// 调用WtDtRunner的parser_unsubscribe方法，传递解析器ID和合约代码
		// 外部模块根据这些信息向数据源发送退订请求
		getRunner().parser_unsubscribe(_id.c_str(), code.c_str());
}
```

### 注册回调接口 registerSpi
```cpp
/**
 * @brief 注册回调接口
 * @param listener 回调接口指针，用于接收行情数据和事件
 * 
 * 该函数保存回调接口指针，并从回调接口获取基础数据管理器。
 * 回调接口用于向系统推送行情数据和事件。
 * 基础数据管理器用于访问合约、交易所等基础信息。
 */
void ExpParser::registerSpi(IParserSpi* listener)
{
	// 保存回调接口指针
	m_sink = listener;

	// 如果回调接口指针有效，则从回调接口获取基础数据管理器
	// 基础数据管理器用于访问合约、交易所、交易时段等基础信息
	if (m_sink)
		m_pBaseDataMgr = m_sink->getBaseDataMgr();
}
```

# 数据服务运行器 WtDtRunner.h/cpp
数据服务运行器。主要功能包括：
- 数据服务的初始化、启动和运行
- 行情解析器（Parser）的创建、配置和运行
- 数据转储器（Dumper）的创建、配置和回调
- 基础数据（合约、交易所、交易时段等）的加载和维护
- 主力合约规则和次主力合约规则的加载
- 数据写入器（DataWriter）的初始化和运行
- 状态监控器（StateMonitor）的初始化和运行
- 数据广播器（UDPCaster、ShmCaster）的初始化和运行
- 指数工厂（IndexFactory）的初始化和运行
- 提供扩展Parser和扩展Dumper的创建和管理接口
- 处理扩展Parser的事件和订阅回调
- 处理扩展Dumper的数据转储回调

设计思想：
- 单例模式：通过getRunner()函数获取全局唯一实例
- 模块化设计：将不同功能分离到不同的管理器中（DataManager、ParserAdapter、StateMonitor等）
- 回调机制：通过函数指针实现扩展Parser和Dumper的回调，支持外部自定义逻辑

## 成员
- `WTSBaseDataMgr	_bd_mgr`：基础数据管理器：管理合约、交易所、交易时段等基础信息
- `WTSHotMgr		_hot_mgr`：主力合约管理器：管理主力合约规则和次主力合约规则
- `boost::asio::io_service _async_io`：Boost异步IO服务：用于异步IO操作
- `StateMonitor	_state_mon`：状态监控器：监控交易时段状态，管理数据存储的打开和关闭
- `UDPCaster		_udp_caster`：UDP广播器：通过UDP协议广播行情数据
- `ShmCaster		_shm_caster`：共享内存广播器：通过共享内存广播行情数据
- `DataManager		_data_mgr`：数据管理器：管理行情数据的接收、处理、存储和分发
- `IndexFactory	_idx_factory`：指数工厂：管理指数的计算和发布
- `ParserAdapterMgr	_parsers`：行情解析器管理器：管理所有行情解析器的运行
- `bool _to_exit`：退出标志：true表示需要退出，false表示继续运行
---
- `FuncParserEvtCallback	_cb_parser_evt`：扩展Parser事件回调函数指针
- `FuncParserSubCallback	_cb_parser_sub`：扩展Parser订阅回调函数指针
---
- `FuncDumpBars	_dumper_for_bars`：K线数据转储回调函数指针
- `FuncDumpTicks	_dumper_for_ticks`：Tick数据转储回调函数指针
- `FuncDumpOrdQue	_dumper_for_ordque`：委托队列数据转储回调函数指针
- `FuncDumpOrdDtl	_dumper_for_orddtl`：委托明细数据转储回调函数指针
- `FuncDumpTrans	_dumper_for_trans`：逐笔成交数据转储回调函数指针
---
- `ExpDumpers		_dumpers`：扩展转储器映射表：管理所有扩展转储器实例
  - typedef std::map\<std::string, ExpDumperPtr\> ExpDumpers：扩展转储器映射表类型
  - typedef std::shared_ptr\<`ExpDumper`\> ExpDumperPtr：扩展转储器智能指针类型


## 方法

### 初始化与生命周期

#### 初始化数据服务 initialize
流程：
- 使用参数 logCfg 和 bLogCfgFile 对日志系统 WTSLogger 进行初始化
- 使用参数 modDir 通过 WtHelper 设置模块目录
- 通过 cfgFile 和 bCfgFile 加载基础数据文件配置到 configs
  - 读取 `basefiles`，使用其中的配置文件路径初始化 `_bd_mgr: WTSBaseDataMgr`、`_hot_mgr: WTSHotMgr`
  - 读取 `shmcaster`、`broadcaster` 来初始化 `_shm_caster: ShmCaster`、`_udp_caster: UDPCaster`，并设置 `_data_mgr: DataManager`
  - 读取 `allday` 来初始化 `_state_mon: StateMonitor`
  - 读取 `writer` 来初始化 `_data_mgr: DataManager`
  - 读取 `index`、`parsers` 来初始化 `_idx_factory: IndexFactory`、`_parsers: ParserAdapterMgr`
```cpp
/**
 * @brief 初始化数据服务
 * @param cfgFile 配置文件路径或配置内容字符串
 * @param logCfg 日志配置文件路径或配置内容字符串
 * @param modDir 模块目录路径，默认为空字符串（使用当前目录）
 * @param bCfgFile cfgFile是否为文件路径，默认为true
 * @param bLogCfgFile logCfg是否为文件路径，默认为true
 */
void WtDtRunner::initialize(const char* cfgFile, const char* logCfg, const char* modDir /* = "" */, bool bCfgFile /* = true */, bool bLogCfgFile /* = true */)
```

#### 启动数据服务 start
```cpp
/**
 * @brief 启动数据服务
 * @param bAsync 是否异步启动，默认为false（同步启动）
 * @param bAlldayMode 是否全天候模式，默认为false（普通模式）
 * 
 * 该函数启动数据服务，开始运行行情解析器和数据管理器。
 * 
 * 同步模式（bAsync=false）：
 * - 安装信号处理钩子，捕获系统信号（如Ctrl+C）
 * - 在异步IO线程池中启动状态监控器（非全天候模式）
 * - 创建一个工作线程循环执行异步IO任务
 * - 阻塞当前线程直到接收到退出信号
 * 
 * 异步模式（bAsync=true）：
 * - 直接启动状态监控器（非全天候模式）
 * - 立即返回，数据服务在后台运行
 * 
 * 全天候模式（bAlldayMode=true）：
 * - 不启动状态监控器
 * - 适用于7x24交易市场（如数字货币）
 */
void WtDtRunner::start(bool bAsync /* = false */, bool bAlldayMode /* = false */)
{
	// 启动所有行情解析器，开始接收和处理行情数据
	_parsers.run();

    if(!bAsync)  // 同步启动模式
    {
		// 安装信号处理钩子，用于捕获系统信号（如SIGINT、SIGTERM）
		install_signal_hooks(
			// 错误消息处理回调：当捕获到信号时，如果未设置退出标志，则记录错误日志
			[this](const char* message) {
				if(!_to_exit)  // 如果尚未设置退出标志
					WTSLogger::error(message);  // 记录错误消息
			}, 
			// 退出标志设置回调：设置退出标志，触发优雅退出
			[this](bool toExit) {
				if (_to_exit)  // 如果已经设置了退出标志
					return;  // 直接返回，避免重复设置
				_to_exit = toExit;  // 设置退出标志
				WTSLogger::info("Exit flag is {}", _to_exit);  // 记录退出标志状态
			});

		// 在异步IO线程池中投递一个任务：启动状态监控器
		_async_io.post([this, bAlldayMode]() {
			if(!bAlldayMode)  // 如果不是全天候模式
			{
				std::this_thread::sleep_for(std::chrono::milliseconds(5));  // 短暂延迟5毫秒
				_state_mon.run();  // 启动状态监控器，监控交易时段状态
			}
		});

		// 创建一个工作线程，循环执行异步IO任务
		StdThread trd([this] {
			while (!_to_exit)  // 当退出标志未设置时，持续运行
			{
				std::this_thread::sleep_for(std::chrono::milliseconds(2));  // 短暂延迟2毫秒，避免CPU占用过高
				_async_io.run_one();  // 执行一个异步IO任务
			}
		});

		trd.join();  // 等待工作线程结束，阻塞当前线程直到接收到退出信号
    }
	else  // 异步启动模式
	{
		if (!bAlldayMode)  // 如果不是全天候模式
		{
			std::this_thread::sleep_for(std::chrono::milliseconds(5));  // 短暂延迟5毫秒
			_state_mon.run();  // 启动状态监控器
		}
		// 异步模式立即返回，数据服务在后台运行
	}
}
```

### 扩展行情解析器接口

#### 创建扩展行情解析器 createExtParser
```cpp
/**
 * @brief 创建扩展行情解析器
 * @param id 解析器唯一标识符
 * @return bool 创建成功返回true
 * 
 * 该函数创建一个扩展行情解析器实例。
 * 扩展解析器用于接入自定义的行情数据源，将外部行情推送到系统中。
 */
bool WtDtRunner::createExtParser(const char* id)
{
	// 创建ParserAdapter实例
	ParserAdapterPtr adapter(new ParserAdapter(&_bd_mgr, &_data_mgr, &_idx_factory));
	// 创建ExpParser实例
	ExpParser* parser = new ExpParser(id);
	// 初始化扩展解析器，将ExpParser与ParserAdapter关联
	adapter->initExt(id, parser);
	// 将适配器添加到管理器
	_parsers.addAdapter(id, adapter);
	WTSLogger::info("Extended parser {} created", id);  // 记录日志
	return true;  // 返回成功
}
```

#### 注册扩展Parser的回调函数 registerParserPorter
```cpp
/**
 * @brief 注册扩展Parser的回调函数
 * @param cbEvt 行情解析器事件回调函数
 * @param cbSub 行情订阅回调函数
 * 
 * 该函数保存扩展Parser的回调函数指针，用于处理解析器事件和订阅请求。
 */
void WtDtRunner::registerParserPorter(FuncParserEvtCallback cbEvt, FuncParserSubCallback cbSub)
{
	_cb_parser_evt = cbEvt;  // 保存事件回调函数指针
	_cb_parser_sub = cbSub;  // 保存订阅回调函数指针
	WTSLogger::info("Callbacks of Extented Parser registration done");  // 记录日志
}
```

#### Parser初始化事件处理 parser_init
```cpp
/**
 * @brief Parser初始化事件处理
 * @param id 解析器ID
 * 
 * 该函数处理Parser的初始化事件，如果注册了事件回调函数，则触发回调。
 */
void WtDtRunner::parser_init(const char* id)
{
	if (_cb_parser_evt)  // 如果注册了事件回调函数
		_cb_parser_evt(EVENT_PARSER_INIT, id);  // 触发初始化事件回调
}
```

#### Parser连接事件处理 parser_connect
```cpp
/**
 * @brief Parser连接事件处理
 * @param id 解析器ID
 * 
 * 该函数处理Parser的连接事件，如果注册了事件回调函数，则触发回调。
 */
void WtDtRunner::parser_connect(const char* id)
{
	if (_cb_parser_evt)  // 如果注册了事件回调函数
		_cb_parser_evt(EVENT_PARSER_CONNECT, id);  // 触发连接事件回调
}
```

#### Parser释放事件处理 parser_release
```cpp
/**
 * @brief Parser释放事件处理
 * @param id 解析器ID
 * 
 * 该函数处理Parser的释放事件，如果注册了事件回调函数，则触发回调。
 */
void WtDtRunner::parser_release(const char* id)
{
	if (_cb_parser_evt)  // 如果注册了事件回调函数
		_cb_parser_evt(EVENT_PARSER_RELEASE, id);  // 触发释放事件回调
}
```

#### Parser断开连接事件处理 parser_disconnect
```cpp
/**
 * @brief Parser断开连接事件处理
 * @param id 解析器ID
 * 
 * 该函数处理Parser的断开连接事件，如果注册了事件回调函数，则触发回调。
 */
void WtDtRunner::parser_disconnect(const char* id)
{
	if (_cb_parser_evt)  // 如果注册了事件回调函数
		_cb_parser_evt(EVENT_PARSER_DISCONNECT, id);  // 触发断开连接事件回调
}
```

#### Parser订阅处理 parser_subscribe
```cpp
/**
 * @brief Parser订阅处理
 * @param id 解析器ID
 * @param code 合约代码
 * 
 * 该函数处理Parser的订阅请求，如果注册了订阅回调函数，则触发回调。
 */
void WtDtRunner::parser_subscribe(const char* id, const char* code)
{
	if (_cb_parser_sub)  // 如果注册了订阅回调函数
		_cb_parser_sub(id, code, true);  // 触发订阅回调，第三个参数true表示订阅
}
```

#### Parser退订处理 parser_unsubscribe
```cpp
/**
 * @brief Parser退订处理
 * @param id 解析器ID
 * @param code 合约代码
 * 
 * 该函数处理Parser的退订请求，如果注册了订阅回调函数，则触发回调。
 */
void WtDtRunner::parser_unsubscribe(const char* id, const char* code)
{
	if (_cb_parser_sub)  // 如果注册了订阅回调函数
		_cb_parser_sub(id, code, false);  // 触发退订回调，第三个参数false表示退订
}
```

#### 处理扩展Parser推送的行情数据 on_ext_parser_quote
```cpp
/**
 * @brief 处理扩展Parser推送的行情数据
 * @param id 解析器ID
 * @param curTick Tick行情数据指针
 * @param uProcFlag 处理标记
 * 
 * 该函数接收扩展Parser推送的行情数据，转发给对应的ParserAdapter进行处理。
 * 如果找不到对应的解析器，记录警告日志。
 */
void WtDtRunner::on_ext_parser_quote(const char* id, WTSTickStruct* curTick, uint32_t uProcFlag)
{
	ParserAdapterPtr adapter = _parsers.getAdapter(id);  // 根据ID获取解析器适配器
	if (adapter)  // 如果找到了对应的适配器
	{
		WTSTickData* newTick = WTSTickData::create(*curTick);  // 创建WTSTickData对象
		adapter->handleQuote(newTick, uProcFlag);  // 处理行情数据
		newTick->release();  // 释放Tick数据对象
	}
	else  // 如果未找到对应的适配器
	{
		WTSLogger::warn("Parser {} not exists", id);  // 记录警告日志
	}
}
```

### 扩展数据转储器接口

#### 创建扩展数据转储器 createExtDumper
```cpp
/**
 * @brief 创建扩展数据转储器
 * @param id 转储器唯一标识符
 * @return bool 创建成功返回true
 * 
 * 该函数创建一个扩展数据转储器实例。
 * 扩展转储器用于将历史数据导出到自定义存储系统。
 */
bool WtDtRunner::createExtDumper(const char* id)
{
	// 创建ExpDumper实例，使用智能指针管理
	ExpDumperPtr dumper(new ExpDumper(id));
	// 将转储器添加到映射表
	_dumpers[id] = dumper;
	// 将转储器注册到数据管理器
	_data_mgr.add_ext_dumper(id, dumper.get());
	WTSLogger::info("Extended dumper {} created", id);  // 记录日志
	return true;  // 返回成功
}
```

#### 注册扩展Dumper的回调函数 (K线和Tick) registerExtDumper
```cpp
/**
 * @brief 注册扩展Dumper的回调函数（K线和Tick）
 * @param barDumper K线数据转储回调函数
 * @param tickDumper Tick数据转储回调函数
 * 
 * 该函数保存扩展Dumper的基础数据转储回调函数指针。
 */
void WtDtRunner::registerExtDumper(FuncDumpBars barDumper, FuncDumpTicks tickDumper)
{
	_dumper_for_bars = barDumper;   // 保存K线转储回调函数指针
	_dumper_for_ticks = tickDumper;  // 保存Tick转储回调函数指针
}
```

#### 注册扩展Dumper的回调函数 (高频数据) registerExtHftDataDumper`
```cpp
/**
 * @brief 注册扩展Dumper的回调函数（高频数据）
 * @param ordQueDumper 委托队列数据转储回调函数
 * @param ordDtlDumper 委托明细数据转储回调函数
 * @param transDumper 逐笔成交数据转储回调函数
 * 
 * 该函数保存扩展Dumper的高频数据转储回调函数指针。
 */
void WtDtRunner::registerExtHftDataDumper(FuncDumpOrdQue ordQueDumper, FuncDumpOrdDtl ordDtlDumper, FuncDumpTrans transDumper)
{
	_dumper_for_ordque = ordQueDumper;  // 保存委托队列转储回调函数指针
	_dumper_for_orddtl = ordDtlDumper;  // 保存委托明细转储回调函数指针
	_dumper_for_trans = transDumper;    // 保存逐笔成交转储回调函数指针
}
```

#### 转储历史K线数据 dumpHisBars
```cpp
/**
 * @brief 转储历史K线数据
 * @param id 转储器ID
 * @param stdCode 标准合约代码
 * @param period K线周期
 * @param bars K线数据数组指针
 * @param count K线数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将历史K线数据转储到外部存储系统。
 * 如果未注册K线转储回调函数，返回false。
 */
bool WtDtRunner::dumpHisBars(const char* id, const char* stdCode, const char* period, WTSBarStruct* bars, uint32_t count)
{
	if (NULL == _dumper_for_bars)  // 如果未注册K线转储回调函数
	{
		WTSLogger::error("Extended bar dumper not enabled");  // 记录错误日志
		return false;  // 返回失败
	}

	// 调用K线转储回调函数，执行实际的转储操作
	return _dumper_for_bars(id, stdCode, period, bars, count);
}
```

#### 转储历史Tick数据 dumpHisTicks
```cpp
/**
 * @brief 转储历史Tick数据
 * @param id 转储器ID
 * @param stdCode 标准合约代码
 * @param uDate 交易日期
 * @param ticks Tick数据数组指针
 * @param count Tick数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将历史Tick数据转储到外部存储系统。
 * 如果未注册Tick转储回调函数，返回false。
 */
bool WtDtRunner::dumpHisTicks(const char* id, const char* stdCode, uint32_t uDate, WTSTickStruct* ticks, uint32_t count)
{
	if (NULL == _dumper_for_ticks)  // 如果未注册Tick转储回调函数
	{
		WTSLogger::error("Extended tick dumper not enabled");  // 记录错误日志
		return false;  // 返回失败
	}

	// 调用Tick转储回调函数，执行实际的转储操作
	return _dumper_for_ticks(id, stdCode, uDate, ticks, count);
}
```

#### 转储历史委托队列数据 dumpHisOrdQue
```cpp
/**
 * @brief 转储历史委托队列数据
 * @param id 转储器ID
 * @param stdCode 标准合约代码
 * @param uDate 交易日期
 * @param items 委托队列数据数组指针
 * @param count 委托队列数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将历史委托队列数据转储到外部存储系统。
 * 如果未注册委托队列转储回调函数，返回false。
 */
bool WtDtRunner::dumpHisOrdQue(const char* id, const char* stdCode, uint32_t uDate, WTSOrdQueStruct* items, uint32_t count)
{
	if (NULL == _dumper_for_ordque)  // 如果未注册委托队列转储回调函数
	{
		WTSLogger::error("Extended order queue dumper not enabled");  // 记录错误日志
		return false;  // 返回失败
	}

	// 调用委托队列转储回调函数，执行实际的转储操作
	return _dumper_for_ordque(id, stdCode, uDate, items, count);
}
```

#### 转储历史委托明细数据 dumpHisOrdDtl
```cpp
/**
 * @brief 转储历史委托明细数据
 * @param id 转储器ID
 * @param stdCode 标准合约代码
 * @param uDate 交易日期
 * @param items 委托明细数据数组指针
 * @param count 委托明细数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将历史委托明细数据转储到外部存储系统。
 * 如果未注册委托明细转储回调函数，返回false。
 */
bool WtDtRunner::dumpHisOrdDtl(const char* id, const char* stdCode, uint32_t uDate, WTSOrdDtlStruct* items, uint32_t count)
{
	if (NULL == _dumper_for_orddtl)  // 如果未注册委托明细转储回调函数
	{
		WTSLogger::error("Extended order detail dumper not enabled");  // 记录错误日志
		return false;  // 返回失败
	}

	// 调用委托明细转储回调函数，执行实际的转储操作
	return _dumper_for_orddtl(id, stdCode, uDate, items, count);
}
```

#### 转储历史逐笔成交数据 dumpHisTrans
```cpp
/**
 * @brief 转储历史逐笔成交数据
 * @param id 转储器ID
 * @param stdCode 标准合约代码
 * @param uDate 交易日期
 * @param items 逐笔成交数据数组指针
 * @param count 逐笔成交数据条数
 * @return bool 转储成功返回true，失败返回false
 * 
 * 该函数将历史逐笔成交数据转储到外部存储系统。
 * 如果未注册逐笔成交转储回调函数，返回false。
 */
bool WtDtRunner::dumpHisTrans(const char* id, const char* stdCode, uint32_t uDate, WTSTransStruct* items, uint32_t count)
{
	if (NULL == _dumper_for_trans)  // 如果未注册逐笔成交转储回调函数
	{
		WTSLogger::error("Extended transaction dumper not enabled");  // 记录错误日志
		return false;  // 返回失败
	}

	// 调用逐笔成交转储回调函数，执行实际的转储操作
	return _dumper_for_trans(id, stdCode, uDate, items, count);
}
```

# 数据服务模块WtDtPorter对外C语言接口 WtDtPorter.h/cpp
定义了WtDtPorter模块对外提供的C语言接口，是WonderTrader数据服务模块的对外API。

## 基础接口

### 初始化数据服务 initialize
```cpp
/**
 * @brief 初始化数据服务
 * @param cfgFile 配置文件路径或配置内容字符串
 * @param logCfg 日志配置文件路径或配置内容字符串
 * @param bCfgFile cfgFile是否为文件路径
 * @param bLogCfgFile logCfg是否为文件路径
 * 
 * 该函数初始化WtDtPorter数据服务。
 * 在Windows平台下，首先启用MiniDumper用于崩溃转储；
 * 然后调用WtDtRunner的初始化方法，加载配置文件和日志配置。
 */
void initialize(WtString cfgFile, WtString logCfg, bool bCfgFile, bool bLogCfgFile)
{
#ifdef _MSC_VER  // Windows平台下启用MiniDumper
	// 启用MiniDumper，程序崩溃时会在当前工作目录生成dump文件
	CMiniDumper::Enable(getModuleName(), true, WtHelper::get_cwd());
#endif
	// 调用WtDtRunner的初始化方法
	// 参数：配置文件、日志配置、模块目录、配置类型标志、日志配置类型标志
	getRunner().initialize(cfgFile, logCfg, getBinDir(), bCfgFile, bLogCfgFile);
}
```

### 启动数据服务 start
```cpp
/**
 * @brief 启动数据服务
 * @param bAsync 是否异步启动，默认为false（同步启动）
 * 
 * 该函数启动数据服务，开始运行行情解析器和数据管理器。
 * 如果bAsync为false，函数会阻塞当前线程，直到接收到退出信号；
 * 如果bAsync为true，函数会立即返回，数据服务在后台运行。
 */
void start(bool bAsync/* = false*/)
{
	// 调用WtDtRunner的启动方法，传递异步标志
	getRunner().start(bAsync);
}
```

## 辅助接口

### 获取版本信息 get_version
```cpp
/**
 * @brief 获取版本信息
 * @return const char* 版本信息字符串
 * 
 * 该函数返回WtDtPorter模块的版本信息。
 * 版本信息包括平台类型、版本号、编译日期和时间。
 * 使用静态字符串缓存版本信息，避免重复构建。
 */
const char* get_version()
{
	static std::string _ver;  // 静态变量缓存版本信息
	if (_ver.empty())  // 如果尚未构建版本信息
	{
		// 构建版本信息字符串
		_ver = PLATFORM_NAME;  // 平台名称：X64/X86/UNIX
		_ver += " ";
		_ver += WT_VERSION;  // 版本号，定义在WTSVersion.h中
		_ver += " Build@";
		_ver += __DATE__;  // 编译日期
		_ver += " ";
		_ver += __TIME__;  // 编译时间
	}
	return _ver.c_str();  // 返回版本信息字符串
}
```

### 输出日志 write_log
```cpp
/**
 * @brief 输出日志
 * @param level 日志级别
 * @param message 日志内容
 * @param catName 日志分类名称
 * 
 * 该函数输出日志到系统日志系统。
 * 如果指定了日志分类名称，则按分类输出；否则按默认方式输出。
 */
void write_log(unsigned int level, const char* message, const char* catName)
{
	if (strlen(catName) > 0)  // 如果指定了日志分类名称
	{
		// 按分类输出日志
		WTSLogger::log_raw_by_cat(catName, (WTSLogLevel)level, message);
	}
	else  // 如果未指定日志分类名称
	{
		// 按默认方式输出日志
		WTSLogger::log_raw((WTSLogLevel)level, message);
	}
}
```

## 扩展行情解析器接口

### 创建扩展行情解析器 create_ext_parser
```cpp
/**
 * @brief 创建扩展行情解析器
 * @param id 解析器唯一标识符
 * @return bool 创建成功返回true，失败返回false
 * 
 * 该函数创建一个扩展行情解析器实例。
 * 调用WtDtRunner的createExtParser方法创建解析器。
 */
bool create_ext_parser(const char* id)
{
	// 调用WtDtRunner的createExtParser方法，传递解析器ID
	return getRunner().createExtParser(id);
}
```

### 注册扩展Parser的回调函数 register_parser_callbacks
```cpp
/**
 * @brief 注册扩展Parser的回调函数
 * @param cbEvt 行情解析器事件回调函数
 * @param cbSub 行情订阅回调函数
 * 
 * 该函数注册扩展Parser的回调函数。
 * 调用WtDtRunner的registerParserPorter方法注册回调函数。
 */
void register_parser_callbacks(FuncParserEvtCallback cbEvt, FuncParserSubCallback cbSub)
{
	// 调用WtDtRunner的registerParserPorter方法，传递事件回调和订阅回调
	getRunner().registerParserPorter(cbEvt, cbSub);
}
```

### 向底层推送tick数据 parser_push_quote
```cpp
/**
 * @brief 向底层推送tick数据
 * @param id 解析器ID
 * @param curTick 最新tick数据指针
 * @param uProcFlag 处理标记
 * 
 * 该函数将外部接收到的tick行情数据推送到系统中。
 * 调用WtDtRunner的on_ext_parser_quote方法处理行情数据。
 */
void parser_push_quote(const char* id, WTSTickStruct* curTick, WtUInt32 uProcFlag)
{
	// 调用WtDtRunner的on_ext_parser_quote方法，传递解析器ID、tick数据和处理标记
	getRunner().on_ext_parser_quote(id, curTick, uProcFlag);
}
```

## 扩展数据转储器接口

### 创建扩展数据转储器 create_ext_dumper
```cpp
/**
 * @brief 创建扩展数据转储器
 * @param id 转储器唯一标识符
 * @return bool 创建成功返回true，失败返回false
 * 
 * 该函数创建一个扩展数据转储器实例。
 * 调用WtDtRunner的createExtDumper方法创建转储器。
 */
bool create_ext_dumper(const char* id)
{
	// 调用WtDtRunner的createExtDumper方法，传递转储器ID
	return getRunner().createExtDumper(id);
}
```

### 注册扩展Dumper的回调函数 (K线和Tick) register_extended_dumper
```cpp
/**
 * @brief 注册扩展Dumper的回调函数（K线和Tick）
 * @param barDumper K线数据转储回调函数
 * @param tickDumper Tick数据转储回调函数
 * 
 * 该函数注册扩展Dumper的基础数据转储回调函数。
 * 调用WtDtRunner的registerExtDumper方法注册回调函数。
 */
void register_extended_dumper(FuncDumpBars barDumper, FuncDumpTicks tickDumper)
{
	// 调用WtDtRunner的registerExtDumper方法，传递K线转储回调和Tick转储回调
	getRunner().registerExtDumper(barDumper, tickDumper);
}
```

### 注册扩展Dumper的回调函数 (高频数据) register_extended_hftdata_dumper
```cpp
/**
 * @brief 注册扩展Dumper的回调函数（高频数据）
 * @param ordQueDumper 委托队列数据转储回调函数
 * @param ordDtlDumper 委托明细数据转储回调函数
 * @param transDumper 逐笔成交数据转储回调函数
 * 
 * 该函数注册扩展Dumper的高频数据转储回调函数。
 * 调用WtDtRunner的registerExtHftDataDumper方法注册回调函数。
 */
void register_extended_hftdata_dumper(FuncDumpOrdQue ordQueDumper, FuncDumpOrdDtl ordDtlDumper, FuncDumpTrans transDumper)
{
	// 调用WtDtRunner的registerExtHftDataDumper方法，传递委托队列、委托明细、逐笔成交转储回调
	getRunner().registerExtHftDataDumper(ordQueDumper, ordDtlDumper, transDumper);
}
```